# File for plots

Read data generated in `reinforcement_learning.ipynb`. Plot and compare. Save plots.

### Imports and settings

In [2]:
# Imports

# General
import numpy as np
import pandas as pd
from matplotlib import pyplot as pyplot
from tqdm import tqdm # used for progress bar
import json
import os

# Specials
from pyworld3 import World3
from pyworld3.utils import plot_world_variables, standard_setup
from rewards import *

In [3]:
# Settings
SAVE_PLOTS = True

### Data extraction

In [4]:
TRAIN_ID = "1"
RL_ID = "100"
CTRLWORLD_ID = "DCFSN"
REWARD_NAME = "doughnut"

In [ ]:
def read_json(ctrlworld_id=None, reward_name=REWARD_NAME, train_id=TRAIN_ID, rl_id=RL_ID):
    # ctrlworld_id: choose None to read from standard world instead of control world

    noise_status = None
    seed = -1
    ctrl_names = []
    grid_size = None
    threshold = None
    lookahead = None

    if ctrlworld_id==None:
        json_path = f"datasets/ctrl_results/{reward_name}/train{train_id}_rl{rl_id}_addinf_std.json"
    else:
        json_path = f"datasets/ctrl_results/{reward_name}/train{train_id}_rl{rl_id}_addinf_ctrlwrld{ctrlworld_id}.json"
    json_file = os.path.join(os.getcwd(), json_path)

    with open(json_file) as fjson:
        table = json.load(fjson)[0]
    
    noise_status = table["noise"]

    if noise_status:
        if ctrlworld_id==None:
            seed = table["seed_gen_std"]
        else:
            seed = table["seed_gen_ctrl"]

    if ctrlworld_id!=None:
        threshold = table["threshold"]
        lookahead = table["lookahead"]
        num_ctrls = table["num_ctrl_funcs"]
        ctrl1 = table["ctrl1"]
        grid_sz1 = table["grid_size_1"]
        ctrl_names.append(ctrl1)
        grid_size = [grid_sz1]
        if num_ctrls > 1:
            ctrl2 = table["ctrl2"]
            grid_sz2 = table["grid_size_2"]
            ctrl_names.append(ctrl2)
            grid_size.append(grid_sz2)

    return noise_status, seed, ctrl_names, grid_size, threshold, lookahead


def read_parquet(control_names=[], ctrlworld_id=None, reward_name=REWARD_NAME, train_id=TRAIN_ID, rl_id=RL_ID, expected_reward_cols=["J", "Jnorm", "nn", "nnnorm", "reward"], state_but_not_init_vars=["time"]):

    ctrl_vals = {}

    if ctrlworld_id!=None:
        parquet_path = f"datasets/ctrl_results/{reward_name}/train{train_id}_rl{rl_id}_control_world{ctrlworld_id}.parquet"
    else:
        parquet_path = f"datasets/ctrl_results/{reward_name}/train{train_id}_rl{rl_id}_standard_world.parquet"
    df = pd.read_parquet(parquet_path)

    col_names = df.columns # https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.columns.html
    state_var_names = [col for col in col_names if (col not in expected_reward_cols)]
    init_state_var_names = [var for var in state_var_names if (var not in state_but_not_init_vars)]

    init_state_var_vals = df[init_state_var_names].to_numpy()[0]
    Js = df["J"].to_numpy()
    Js_norm = df["Jnorm"].to_numpy()
    nn_approx = df["nn"].to_numpy()
    nn_approx_norm = df["nnnorm"].to_numpy()
    reward = df["reward"].to_numpy()

    if ctrlworld_id!=None:
        for control_name in control_names:
            current_ctrl_vals = df[f"{control_name}_control"].to_numpy()
            ctrl_vals[control_name] = current_ctrl_vals
    
    times = df["time"].to_numpy()
    start_year = int(times[0])
    end_year = int(times[-1])

    return state_var_names, init_state_var_vals, Js, Js_norm, nn_approx, nn_approx_norm, reward, ctrl_vals, start_year, end_year


def rebuild_world(ctrlworld_id=None, reward_name=REWARD_NAME, train_id=TRAIN_ID, rl_id=RL_ID):
    # Returns: World3 object along with J, Jnorm, nn, nnnorm and reward

    # Step 1: Get data from JSON file
    noise_status, seed, ctrl_names, grid_size, threshold, lookahead = read_json(ctrlworld_id, reward_name, train_id, rl_id)

    # Step 2: Get data from parquet file
    state_var_names, init_state_var_vals, Js, Js_norm, nn_approx, nn_approx_norm, reward, ctrl_vals, start_year, end_year = read_parquet(control_names=ctrl_names, ctrlworld_id=ctrlworld_id, reward_name=reward_name, train_id=train_id, rl_id=rl_id)

    # Step 3: Build world
    world = World3(year_min=start_year, year_max=end_year, noise=noise_status, seed=seed)
    world.set_world3_control()
    world.init_world3_constants()
    world.init_world3_variables()
    world.set_world3_table_functions()
    world.set_world3_noise_stds()
    world.set_world3_delay_functions()
    world.run_world3(fast=False)


    return

def rebuild_control():
    return

def rebuild_standard():
    return

1900 2100


In [ ]:
rewards = {}
Js = {}
Js_norm = {}
nn_approx = {}
nn_approx_norm = {}

### Plot